In [0]:
base_path = "/Volumes/workspace/default/my_volume/flight_project"
gold_airline_path = f"{base_path}/gold/airline_summary"
gold_cascade_path = f"{base_path}/gold/cascade_impact"
silver_path       = f"{base_path}/silver/flights"

print("Paths set!")

Paths set!


In [0]:
# Spark SQL mein tables register karo
spark.read.format("delta").load(silver_path) \
    .createOrReplaceTempView("silver_flights")

spark.read.format("delta").load(gold_airline_path) \
    .createOrReplaceTempView("gold_airline_summary")

spark.read.format("delta").load(gold_cascade_path) \
    .createOrReplaceTempView("gold_cascade_impact")

print("✅ Tables registered!")

✅ Tables registered!


In [0]:
spark.sql("""
    SELECT 
        COUNT(*) as total_flights,
        SUM(CASE WHEN is_delayed = true THEN 1 ELSE 0 END) as delayed_flights,
        SUM(CASE WHEN is_delayed = false THEN 1 ELSE 0 END) as on_time_flights,
        ROUND(SUM(CASE WHEN is_delayed = true THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as delay_percentage,
        ROUND(AVG(delay_minutes), 2) as avg_delay_minutes
    FROM silver_flights
""").show()

+-------------+---------------+---------------+----------------+-----------------+
|total_flights|delayed_flights|on_time_flights|delay_percentage|avg_delay_minutes|
+-------------+---------------+---------------+----------------+-----------------+
|        12073|           1508|          10565|           12.49|             4.83|
+-------------+---------------+---------------+----------------+-----------------+



In [0]:
spark.sql("""
    SELECT 
        airline_code,
        total_flights,
        delayed_flights,
        avg_delay_minutes,
        delay_percentage
    FROM gold_airline_summary
    ORDER BY delay_percentage DESC
    LIMIT 10
""").show()

+------------+-------------+---------------+-----------------+----------------+
|airline_code|total_flights|delayed_flights|avg_delay_minutes|delay_percentage|
+------------+-------------+---------------+-----------------+----------------+
|         ASI|           14|              9|            28.93|           64.29|
|         NDU|           21|             13|            27.86|            61.9|
|         VAR|           15|              9|             27.0|            60.0|
|         CXK|           63|             32|            22.86|           50.79|
|         N32|           34|             14|            18.53|           41.18|
|         N78|           34|             14|            17.06|           41.18|
|         N59|           17|              7|            18.53|           41.18|
|         CAP|           22|              9|            18.41|           40.91|
|         N82|           30|             12|             18.0|            40.0|
|         N77|           33|            

In [0]:
spark.sql("""
    SELECT
        flight_phase,
        COUNT(*) as total_flights,
        SUM(CASE WHEN is_delayed = true THEN 1 ELSE 0 END) as delayed,
        ROUND(SUM(CASE WHEN is_delayed = true THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as delay_rate
    FROM silver_flights
    GROUP BY flight_phase
    ORDER BY delay_rate DESC
""").show()

+-------------+-------------+-------+----------+
| flight_phase|total_flights|delayed|delay_rate|
+-------------+-------------+-------+----------+
|     APPROACH|         2082|   1476|     70.89|
|       CRUISE|         4812|     32|      0.67|
|CLIMB_DESCENT|         2780|      0|      0.00|
|      LANDING|         2399|      0|      0.00|
+-------------+-------------+-------+----------+



In [0]:
spark.sql("""
    SELECT
        nearest_airport,
        COUNT(*) as delayed_flights,
        SUM(flights_potentially_affected) as total_affected_flights,
        ROUND(AVG(delay_minutes), 2) as avg_delay_minutes
    FROM gold_cascade_impact
    GROUP BY nearest_airport
    ORDER BY total_affected_flights DESC
    LIMIT 10
""").show()

+---------------+---------------+----------------------+-----------------+
|nearest_airport|delayed_flights|total_affected_flights|avg_delay_minutes|
+---------------+---------------+----------------------+-----------------+
|            CGZ|             20|                   340|             45.0|
|            GVT|             16|                   272|             45.0|
|            MQY|             10|                   210|             42.5|
|            BXK|             10|                   200|             45.0|
|            RBD|              7|                   189|            37.86|
|            BOI|             10|                   160|             42.5|
|            TYS|              8|                   160|             45.0|
|            YYC|             10|                   160|             45.0|
|            DAB|              3|                   159|             45.0|
|            SLC|             14|                   154|            43.21|
+---------------+--------

In [0]:
spark.sql("""
    SELECT
        weather_main,
        COUNT(*) as total_flights,
        SUM(CASE WHEN is_delayed = true THEN 1 ELSE 0 END) as delayed_flights,
        ROUND(SUM(CASE WHEN is_delayed = true THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as delay_rate
    FROM silver_flights
    WHERE weather_main IS NOT NULL
    GROUP BY weather_main
    ORDER BY delay_rate DESC
""").show()

+------------+-------------+---------------+----------+
|weather_main|total_flights|delayed_flights|delay_rate|
+------------+-------------+---------------+----------+
|       Clear|           56|              7|     12.50|
|      Clouds|          101|             10|      9.90|
|        Rain|            1|              0|      0.00|
+------------+-------------+---------------+----------+



In [0]:
spark.sql("""
    SELECT
        origin_country,
        COUNT(*) as total_flights,
        SUM(CASE WHEN is_delayed = true THEN 1 ELSE 0 END) as delayed_flights,
        ROUND(SUM(CASE WHEN is_delayed = true THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as delay_rate
    FROM silver_flights
    GROUP BY origin_country
    ORDER BY delayed_flights DESC
    LIMIT 10
""").show()

+--------------+-------------+---------------+----------+
|origin_country|total_flights|delayed_flights|delay_rate|
+--------------+-------------+---------------+----------+
| United States|         7316|           1210|     16.54|
|        Canada|          424|             43|     10.14|
|       Germany|          307|             26|      8.47|
|        France|          177|             21|     11.86|
|United Kingdom|          465|             18|      3.87|
|        Brazil|          102|             15|     14.71|
|        Turkey|          254|             14|      5.51|
|        Mexico|          140|             12|      8.57|
|       Austria|          133|             11|      8.27|
|         Spain|          174|             10|      5.75|
+--------------+-------------+---------------+----------+



In [0]:
%sql
CREATE OR REPLACE VIEW flight_silver AS
SELECT * FROM delta.`/Volumes/workspace/default/my_volume/flight_project/silver/flights`;

CREATE OR REPLACE VIEW flight_gold_airline AS
SELECT * FROM delta.`/Volumes/workspace/default/my_volume/flight_project/gold/airline_summary`;

CREATE OR REPLACE VIEW flight_gold_cascade AS
SELECT * FROM delta.`/Volumes/workspace/default/my_volume/flight_project/gold/cascade_impact`;